# KAN Lyapunov Training


In [ ]:
import math
import timeit
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from kan import KAN
from dreal import Config, Variable, sin
from lyapunov_verification import CheckLyapunov

device = torch.device("cpu")


In [ ]:
def collect_edge_samples(model, x_sample: torch.Tensor):
    model.eval()
    x_sample = x_sample.to(next(model.parameters()).device)
    with torch.no_grad():
        model.get_act(x_sample)
    return (model.spline_preacts, model.spline_postacts)

def fit_edge_polynomials(model, x_sample: torch.Tensor, degree: int=3):
    spline_preacts, spline_postacts = collect_edge_samples(model, x_sample)
    poly_dict = {}
    depth = model.depth
    for l in range(depth):
        pre = spline_preacts[l].cpu().numpy()
        post = spline_postacts[l].cpu().numpy()
        N, out_dim, in_dim = pre.shape
        for j in range(out_dim):
            for i in range(in_dim):
                xs = pre[:, j, i]
                ys = post[:, j, i]
                mask = np.isfinite(xs) & np.isfinite(ys)
                xs_clean = xs[mask]
                ys_clean = ys[mask]
                if xs_clean.size < degree + 1:
                    coeffs = np.array([0.0])
                else:
                    coeffs = np.polyfit(xs_clean, ys_clean, degree)
                poly_dict[l, i, j] = coeffs
    return poly_dict

def poly_to_dreal_expr(coeffs: np.ndarray, x_expr):
    deg = len(coeffs) - 1
    expr = 0.0
    for k, a in enumerate(coeffs):
        power = deg - k
        a_float = float(a)
        if abs(a_float) < 1e-15:
            continue
        if power == 0:
            term = a_float
        elif power == 1:
            term = a_float * x_expr
        else:
            term = a_float * x_expr ** power
        expr = expr + term
    return expr

def build_dreal_kan_from_poly(model, poly_dict, var_names=None):
    device = next(model.parameters()).device
    if var_names is None:
        in_dim = model.width_in[0]
        var_names = [f'x{i + 1}' for i in range(in_dim)]
    else:
        in_dim = len(var_names)
        assert in_dim == model.width_in[0], 'The number of var_names must match the model input dimension.'
    in_vars = [Variable(name) for name in var_names]
    x_layer = in_vars
    for l in range(model.depth):
        in_dim_layer = model.width_in[l]
        out_sub_dim = model.width_out[l + 1]
        assert in_dim_layer == len(x_layer), f'layer {l}: in_dim_layer={in_dim_layer}, but len(x_layer)={len(x_layer)}'
        y_sub = []
        for j in range(out_sub_dim):
            s_j = 0.0
            for i in range(in_dim_layer):
                coeffs = poly_dict[l, i, j]
                s_j = s_j + poly_to_dreal_expr(coeffs, x_layer[i])
            sub_scale = float(model.subnode_scale[l][j].detach().cpu().numpy())
            sub_bias = float(model.subnode_bias[l][j].detach().cpu().numpy())
            s_j = sub_scale * s_j + sub_bias
            y_sub.append(s_j)
        n_sum, n_mult = model.width[l + 1]
        assert n_mult == 0, 'build_dreal_kan_from_poly supports networks without multiplication nodes (width[layer_id][1] == 0). Extend this function to support multiplication nodes.'
        assert n_sum == out_sub_dim
        next_dim = model.width_in[l + 1]
        assert next_dim == n_sum
        x_next = []
        for k in range(next_dim):
            val = y_sub[k]
            node_scale = float(model.node_scale[l][k].detach().cpu().numpy())
            node_bias = float(model.node_bias[l][k].detach().cpu().numpy())
            val = node_scale * val + node_bias
            x_next.append(val)
        x_layer = x_next
    out_exprs = x_layer
    return (in_vars, out_exprs)

def AddCounterexamples1(x: torch.Tensor, CE, N: int, repeat: int=1):
    device = x.device
    dtype = x.dtype
    dim = CE.size()
    dim_samples = []
    for i in range(dim):
        lb = float(CE[i].lb())
        ub = float(CE[i].ub())
        samples_1d = np.random.uniform(lb, ub, size=(N,))
        dim_samples.append(samples_1d)
    nearby_np = np.stack(dim_samples, axis=1)
    if repeat > 1:
        nearby_np = np.repeat(nearby_np, repeats=repeat, axis=0)
    nearby_torch = torch.tensor(nearby_np, dtype=dtype, device=device)
    x_new = torch.cat([x, nearby_torch], dim=0)
    return x_new

def sample_in_annulus(num_samples, ball_lb, ball_ub, device):
    theta = 2 * math.pi * torch.rand(num_samples, device=device)
    r2 = torch.rand(num_samples, device=device) * (ball_ub ** 2 - ball_lb ** 2) + ball_lb ** 2
    r = torch.sqrt(r2)
    x1 = r * torch.cos(theta)
    x2 = r * torch.sin(theta)
    x = torch.stack([x1, x2], dim=1)
    return x


In [ ]:
def V_and_LV_with_factor(model, xb, f_value):
    V_raw = model(xb)
    x1 = xb[:, 0]
    x2 = xb[:, 1]
    factor = (x1 ** 2 + x2 ** 2).unsqueeze(1)
    V_total = factor * V_raw
    f_val = f_value(xb)
    grad_V = torch.autograd.grad(V_total.sum(), xb, create_graph=True)[0]
    L_V = (grad_V * f_val).sum(dim=1, keepdim=True)
    return (V_total, L_V)

def trim_ce_pool(x_all: torch.Tensor, N0: int, max_ce: int) -> torch.Tensor:
    total = x_all.size(0)
    ce_num = total - N0
    if ce_num <= max_ce:
        return x_all
    x_base = x_all[:N0]
    x_ce_all = x_all[N0:]
    x_ce_keep = x_ce_all[-max_ce:] if max_ce > 0 else x_ce_all[:0]
    x_new = torch.cat([x_base, x_ce_keep], dim=0)
    return x_new

def quick_sampling_check_all(model, f_value, ball_lb, ball_ub, num_samples=2000, eps_V=0.0, eps_dV=0.0, device=torch.device('cpu')):
    model.eval()
    xb = sample_in_annulus(num_samples, ball_lb, ball_ub, device)
    xb = xb.detach().requires_grad_(True)
    V_candidate, L_V = V_and_LV_with_factor(model, xb, f_value)
    V = V_candidate.view(-1)
    LV = L_V.view(-1)
    bad_idx = torch.where((V < -eps_V) | (LV > eps_dV))[0]
    if bad_idx.numel() == 0:
        return None
    bad_points = xb[bad_idx].detach().cpu()
    return bad_points


In [ ]:
def advance_shuffle_state(loader):
    """Preserve the original DataLoader shuffle RNG progression."""
    iter(loader)
    for _ in loader.batch_sampler:
        pass


In [ ]:
ce_history = []

def log_ce(points, epoch, ball_ub, source):
    import torch
    if isinstance(points, torch.Tensor):
        pts = points.detach().cpu().numpy()
    else:
        pts = np.asarray(points)
    for i in range(pts.shape[0]):
        ce_history.append({'epoch': int(epoch), 'ball_ub': float(ball_ub), 'source': source, 'x1': float(pts[i, 0]), 'x2': float(pts[i, 1])})

def box_midpoint(box, var_list):
    xs = []
    for v in var_list:
        interval = box[v]
        lb = float(interval.lb())
        ub = float(interval.ub())
        xs.append(0.5 * (lb + ub))
    return np.array(xs, dtype=np.float32)


In [ ]:
def f_value(x):
    x_0 = x[:, 0]
    x_1 = x[:, 1]
    f1 = -1.14045842 * x_0 - 0.22802382 * x_1 + 0.7093115 * x_0 * x_1 ** 2 + 0.49013606 * torch.sin(x_0) - 0.93472954 * x_0 ** 2 * x_1
    f2 = -0.22802382 * x_0 - 1.31777801 * x_1 + 0.51827385 * x_0 ** 2 * x_1 - 0.69488369 * torch.sin(x_1)
    f = torch.stack([f1, f2], dim=1)
    return f


In [ ]:
# Initial sampling box: [-a, a]^2.
N = 500
a = 4
x_all = torch.empty(N, 2).uniform_(-a, a)
x_all = x_all.to(device)
N0 = x_all.size(0)
MAX_CE = 500
alpha = 1.0
config = Config()
config.use_polytope_in_forall = True
config.use_local_optimization = True
config.precision = 0.01
epsilon = 0
ball_lb = 0.05
ball_ub = 2
shrink_r_list = [2, 0.5, 0.1]
shrink_idx = 0
expand_step = 0.05
max_expand_radius = 2.0
phase = 'shrink'
best_radius = None


In [ ]:
valid = False
start = timeit.default_timer()
model = KAN(width=[2, 2, 1], grid=5, k=10, grid_eps=1.0).to(device)
model.train()
t = 0
batch_size = 128
loader = DataLoader(TensorDataset(x_all), batch_size=batch_size, shuffle=True, drop_last=False)
opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-06)
epochs = 30000
patience_per_radius = 300
epochs_at_current_radius = 0
success_at_current_radius = False
sampling_smt_threshold = 100
for epoch in range(1, epochs + 1):
    epochs_at_current_radius += 1
    x_all = x_all.float()
    running = 0.0
    seen = 0
    model.train()
    for xb, in loader:
        xb = xb.detach().requires_grad_(True)
        with torch.no_grad():
            norm_sq = xb.square().sum(dim=1, keepdim=True)
        V_candidate, L_V = V_and_LV_with_factor(model, xb, f_value)
        loss = (
            F.relu(-V_candidate)
            + F.relu(L_V)
            + (norm_sq - alpha * V_candidate).square()
        ).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        running += loss.item() * xb.size(0)
        seen += xb.size(0)
    advance_shuffle_state(loader)
    epoch_loss = running / seen
    if epoch % 10 == 0:
        print(f'epoch {epoch:4d} | loss {epoch_loss:.4e}')
    if epoch % 20 == 0:
        print(f'\n[epoch {epoch}] Verification radius ball_ub = {ball_ub:.3f}, phase = {phase}, epochs at this radius = {epochs_at_current_radius}')
        bad_points = quick_sampling_check_all(model, f_value, ball_lb=ball_lb, ball_ub=ball_ub, num_samples=2000, eps_V=0.0001, eps_dV=0.001, device=device)
        need_smt = False
        if bad_points is not None:
            print('Sampled counterexamples found:', bad_points.shape[0])
            bad_points = bad_points.to(device)
            MAX_NEW_CE = 10
            if bad_points.size(0) > MAX_NEW_CE:
                idx = torch.randperm(bad_points.size(0), device=device)[:MAX_NEW_CE]
                bad_points = bad_points[idx]
            repeat = 1
            ce_repeat = bad_points.repeat(repeat, 1)
            log_ce(ce_repeat, epoch=epoch, ball_ub=ball_ub, source='sampling')
            x_all = torch.cat([x_all, ce_repeat], dim=0)
            x_all = trim_ce_pool(x_all, N0, MAX_CE)
            x_all = x_all.to(device).float()
            loader = DataLoader(TensorDataset(x_all), batch_size=batch_size, shuffle=True, drop_last=False)
            print('Total training samples:', x_all.size(0), '  Counterexample samples:', x_all.size(0) - N0)
            if epochs_at_current_radius >= sampling_smt_threshold:
                print(f'[pre-check] Counterexamples persist within radius {ball_ub:.3f} after {epochs_at_current_radius} epochs at this radius. Running dReal verification.')
                need_smt = True
            else:
                need_smt = False
        else:
            print('[pre-check] No counterexamples found by random sampling. Running dReal verification.')
            need_smt = True
        if not need_smt:
            continue
        degree = 3
        poly_dict = fit_edge_polynomials(model, x_sample=x_all, degree=degree)
        in_vars, out_exprs = build_dreal_kan_from_poly(model, poly_dict, var_names=['x1', 'x2'])
        x1, x2 = in_vars
        V_raw_expr = out_exprs[0]
        factor_expr = x1 ** 2 + x2 ** 2
        V_expr = factor_expr * V_raw_expr
        f1 = -1.14045842 * x1 - 0.22802382 * x2 + 0.7093115 * x1 * x2 ** 2 + 0.49013606 * sin(x1) - 0.93472954 * x1 ** 2 * x2
        f2 = -0.22802382 * x1 - 1.31777801 * x2 + 0.51827385 * x1 ** 2 * x2 - 0.69488369 * sin(x2)
        f = [f1, f2]
        print('===========Verifying==========')
        start_ = timeit.default_timer()
        result = CheckLyapunov(in_vars, f, V_expr, ball_lb, ball_ub, config, epsilon)
        stop_ = timeit.default_timer()
        if result:
            print('Not a Lyapunov function. Found counterexample: ')
            print(result)
            ce_np = box_midpoint(result, in_vars)
            ce_tensor = torch.from_numpy(ce_np[None, :])
            log_ce(ce_tensor, epoch=epoch, ball_ub=ball_ub, source='dreal')
            x_all = AddCounterexamples1(x_all, result, 10, repeat=2)
            x_all = trim_ce_pool(x_all, N0, MAX_CE)
            x_all = x_all.to(device).float()
            loader = DataLoader(TensorDataset(x_all), batch_size=batch_size, shuffle=True, drop_last=False)
            print('Total training samples:', x_all.size(0), '  Counterexample samples:', x_all.size(0) - N0)
            if epochs_at_current_radius >= patience_per_radius and (not success_at_current_radius):
                if phase == 'shrink':
                    shrink_idx += 1
                    if shrink_idx < len(shrink_r_list):
                        old_r = ball_ub
                        ball_ub = shrink_r_list[shrink_idx]
                        epochs_at_current_radius = 0
                        success_at_current_radius = False
                        print(f'[shrink] SMT verification has not passed at radius {old_r:.3f} after {patience_per_radius} epochs. Shrinking to {ball_ub:.3f} and continuing training.')
                    else:
                        print('[shrink] Verification has not passed at any preset radius. No smaller radius is available. Stopping training.')
                        valid = False
                        break
                else:
                    if best_radius is not None:
                        print(f'[expand] SMT verification has not passed at radius {ball_ub:.3f} after {patience_per_radius} epochs. Best verified radius: {best_radius:.3f}.')
                    else:
                        print(f'[expand] SMT verification has not passed at radius {ball_ub:.3f} after {patience_per_radius} epochs. No verified radius has been recorded.')
                    valid = True
                    break
        else:
            print('Satisfy conditions!!')
            success_at_current_radius = True
            epochs_at_current_radius = 0
            if phase == 'shrink':
                best_radius = ball_ub
                phase = 'expand'
                next_r = min(best_radius + expand_step, max_expand_radius)
                print(f'[shrink] First verified radius r0 = {best_radius:.3f}. Switching to expansion; next radius: {next_r:.3f}.')
                ball_ub = next_r
                success_at_current_radius = False
                epochs_at_current_radius = 0
            else:
                best_radius = ball_ub
                next_r = ball_ub + expand_step
                if next_r > max_expand_radius + 1e-09:
                    print(f'[expand] Verification passed at the maximum configured radius {ball_ub:.3f}. Expansion complete.')
                    valid = True
                    break
                else:
                    print(f'[expand] Verification passed at radius {ball_ub:.3f}. Expanding to {next_r:.3f}.')
                    ball_ub = next_r
                    success_at_current_radius = False
                    epochs_at_current_radius = 0
        t += stop_ - start_
        print('==============================')
        continue
stop = timeit.default_timer()
print('\n')
print('Total time: ', stop - start)
print('Verified time: ', t)
model.eval()


In [ ]:
if len(ce_history) > 0:
    x1 = np.array([c['x1'] for c in ce_history])
    x2 = np.array([c['x2'] for c in ce_history])
    ep = np.array([c['epoch'] for c in ce_history])
    r = np.sqrt(x1 ** 2 + x2 ** 2)
    plt.figure(figsize=(6, 6))
    sc = plt.scatter(x1, x2, c=ep, s=20, cmap='viridis')
    plt.colorbar(sc, label='epoch')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('Counterexamples during training')
    plt.axis('equal')
    R = float(ball_ub)
    theta = np.linspace(0, 2 * np.pi, 200)
    plt.plot(R * np.cos(theta), R * np.sin(theta), 'r--', label=f'ball_ub={R}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.figure()
    plt.plot(ep, r, '.-')
    plt.xlabel('epoch')
    plt.ylabel('||x_ce||')
    plt.title('Radius of counterexamples')
    plt.grid(True)
    plt.show()
else:
    print('No counterexamples were recorded during training; ce_history is empty.')
